# Day 08 — Solutions: Files, JSON/CSV, Context Managers
Runnable implementations for safe JSON loader and CSV↔JSON converters.

In [ ]:
# Exercise 1 — Safe JSON loader
from pathlib import Path
from typing import Any
import json

def safe_load_json(path: str | Path, *, verbose: bool = True) -> Any | None:
    p = Path(path)
    try:
        with p.open(encoding='utf-8') as f:
            return json.load(f)
    except FileNotFoundError:
        if verbose: print(f'safe_load_json: not found: {p}')
    except PermissionError:
        if verbose: print(f'safe_load_json: permission denied: {p}')
    except json.JSONDecodeError as e:
        if verbose: print(f'safe_load_json: invalid JSON in {p}: line {e.lineno}, col {e.colno}: {e.msg}')
    return None

# Example (will print if path missing)
safe_load_json('missing.json')

In [ ]:
# Exercise 2 — CSV ↔ JSON converters
import csv

def csv_to_json(csv_path: Path, json_path: Path) -> None:
    with csv_path.open(encoding='utf-8', newline='') as f:
        reader = csv.DictReader(f)
        rows = list(reader)
    with json_path.open('w', encoding='utf-8') as out:
        json.dump(rows, out, indent=2, ensure_ascii=False)

from typing import Any
def json_to_csv(json_path: Path, csv_path: Path) -> None:
    data: list[dict[str, Any]] = safe_load_json(json_path) or []
    if not data:
        csv_path.write_text('', encoding='utf-8')
        return
    fieldnames: list[str] = sorted({k for rec in data for k in rec.keys()})
    with csv_path.open('w', encoding='utf-8', newline='') as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        for rec in data:
            w.writerow({k: rec.get(k, '') for k in fieldnames})